## In this notebook, we will

Use actual data from two ALBATROS antennas

1. Learn how to correlate Electric fields from two antennas to produce visibilities
2. Average them to reduce thermal noise
3. Make phase plots
4. See fringe pattern because of source moving.

In [ ]:
import numpy as np
import numba as nb
import matplotlib.pyplot as plt

In [ ]:
# VISIBILITY FUNCTIONS (trivially parallelizable using Numba JITing)

@nb.njit(parallel=True)
def apply_delay(arr, out, delay, freqs):
    # apply delay to an array of complex electric field or their correlation
    # does exp( j 2 pi nu tau) sign of tau is user dependent
    # freqs should correspond to the columns of the nspec x nchan array
    nspec = arr.shape[0]
    nchan = arr.shape[1]
    for i in nb.prange(nspec):
        for j in range(nchan):
            out[i, j] = arr[i, j] * np.exp(2j * np.pi * freqs[j] * delay[i])
    return out

@nb.njit(parallel=True)
def xcorr_avg(arr1,arr2,acclen):
    # correlate and average, with acclen averaging length
    nblocks = arr1.shape[0]//acclen
    nchan = arr1.shape[1]
    out = np.zeros((nblocks,nchan),dtype=arr1.dtype)
    for i in nb.prange(nblocks):
        for j in range(acclen):
            for k in range(nchan):
                out[i,k] += arr1[i*acclen + j,k]*np.conj(arr2[i*acclen + j,k])
        out[i,:]/=acclen
    return out

Our signal is from the METEOR M2-4 satellite, which transmits at about 137 MHz, with bandwidth of about 100 kHz.

In [ ]:
disk = np.load("../data/summer_school.npz")
data1 = disk['MARS1']
data2 = disk['MARS2']
delays = disk['delays']
freqs = disk['freqs']

In [ ]:
T_SPECTRA = 2*4096/250e6 # we One spectrum from our antenna at this interval
acclen = 2000
integration_time = acclen* T_SPECTRA
nspectra = data1.shape[0]
print('Spectrum Period', T_SPECTRA)
print('Integration Time', integration_time)
print('Total number of spectra', nspectra)

In [ ]:
times = np.arange(0, nspectra)*T_SPECTRA
phase = np.unwrap(np.angle(np.exp(2j*np.pi*delays*freqs[0])))
plt.plot(times, phase)
plt.xlabel('Time (s)')
plt.ylabel('Phase (radians)')
plt.suptitle(f'Phase Evolution of METEOR M2-4 at {freqs[0]/1e6:.2f} MHz')
plt.tight_layout()

### Calculate the rate of change of delay. Is this an astronomical source?

In [ ]:
vis = xcorr_avg(data1, data2, acclen)
im = plt.imshow(np.angle(vis)[:,:], cmap='RdBu', aspect='auto', interpolation='none', vmin=-np.pi, vmax=np.pi)
cbar=plt.colorbar(im)
plt.suptitle(f'Visibility Angle, Unphased, Integration Time {int(integration_time*1e3)} ms')
plt.tight_layout()

In [ ]:
data2_delayed = np.zeros_like(data2)
data2_delayed = apply_delay(data2, data2_delayed, -delays, freqs)

In [ ]:
vis = xcorr_avg(data1, data2_delayed, acclen)
im = plt.imshow(np.angle(vis)[:,:], cmap='RdBu', aspect='auto', interpolation='none', vmin=-np.pi, vmax=np.pi)
cbar=plt.colorbar(im)
plt.suptitle(f'Visibility Angle, Phased, Integration Time {int(integration_time*1e3)} ms')
plt.tight_layout()